In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Lasso, HuberRegressor
from sklearn.metrics import r2_score

In [2]:
df = pd.read_excel("Laptop Prices.xlsx")
df.head()

,Col_0,Col_1,Col_2,Col_3,Col_4,Col_5,Col_6,Col_7,Col_8,Col_9,Col_10
0,Brand,Processor,RAM (GB),Storage,GPU,Screen Size (inch),Resolution,Battery Life (hours),Weight (kg),Operating System,Price ($)
1,apple,8,64,3,Nvidia GTX 1650,17.3,2,8.9,1.42,FreeDOS,3997.07
2,RAZER,4,4,2,Nvidia RTX 3080,14,4,9.4,2.57,Linux,1355.78
3,As us,6,32,1,Nvidia RTX 3060,13.3,1,8.5,1.74,FreeDOS,2673.07
4,lenovo,6,4,4,Nvidia RTX 3080,13.3,4,10.5,3.1,Windows,751.17


In [6]:
df.columns = df.iloc[0]   
df = df[1:]               
df.reset_index(drop=True, inplace=True)

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11767 entries, 0 to 11766
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   Brand                 11767 non-null  object
 1   Processor             11767 non-null  object
 2   RAM (GB)              11767 non-null  object
 3   Storage               11767 non-null  object
 4   GPU                   11767 non-null  object
 5   Screen Size (inch)    11767 non-null  object
 6   Resolution            11767 non-null  object
 7   Battery Life (hours)  11767 non-null  object
 8   Weight (kg)           11767 non-null  object
 9   Operating System      11767 non-null  object
 10  Price ($)             11767 non-null  object
dtypes: object(11)
memory usage: 1011.4+ KB


In [18]:
df["Processor"] = df["Processor"].astype(int)
df["RAM (GB)"] = df["RAM (GB)"].astype(int)
df["Screen Size (inch)"] = df["Screen Size (inch)"].astype(float)
df["Battery Life (hours)"] = df["Battery Life (hours)"].astype(float)
df["Weight (kg)"] = df["Weight (kg)"].astype(float)
df["Price ($)"] = df["Price ($)"].astype(float)

In [8]:
df.isna().sum()

0
Brand                   0
Processor               0
RAM (GB)                0
Storage                 0
GPU                     0
Screen Size (inch)      0
Resolution              0
Battery Life (hours)    0
Weight (kg)             0
Operating System        0
Price ($)               0
dtype: int64

In [9]:
df = df.dropna()


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11767 entries, 0 to 11766
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Brand                 11767 non-null  object 
 1   Processor             11767 non-null  int64  
 2   RAM (GB)              11767 non-null  int64  
 3   Storage               11767 non-null  object 
 4   GPU                   11767 non-null  object 
 5   Screen Size (inch)    11767 non-null  float64
 6   Resolution            11767 non-null  object 
 7   Battery Life (hours)  11767 non-null  float64
 8   Weight (kg)           11767 non-null  float64
 9   Operating System      11767 non-null  object 
 10  Price ($)             11767 non-null  float64
 11  ram_x_cpu             11767 non-null  object 
dtypes: float64(4), int64(2), object(6)
memory usage: 1.1+ MB


In [20]:
X = df.drop('Price ($)', axis=1)
y = df['Price ($)']

In [21]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
model = LinearRegression()

## =================================================

In [23]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    if not outliers.empty:
        print(f"Outliers found in column '{col}':")
        display(outliers[[col, 'Price ($)']])
    else:
        print(f"No outliers found in column '{col}'.")

No outliers found in column 'Processor'.
No outliers found in column 'RAM (GB)'.
No outliers found in column 'Screen Size (inch)'.
No outliers found in column 'Battery Life (hours)'.
No outliers found in column 'Weight (kg)'.
Outliers found in column 'Price ($)':


,Price ($),Price ($)
8,6409.03,6409.03
43,5373.59,5373.59
64,5768.75,5768.75
102,5173.63,5173.63
114,4857.55,4857.55
...,...,...
11664,5110.17,5110.17
11685,9419.75,9419.75
11709,6741.64,6741.64
11720,6402.40,6402.40


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
df['ram_x_cpu'] = df['RAM (GB)'] * df['Processor']
X = df.drop('Price ($)', axis=1)
y = df['Price ($)']
numerical_cols = X.select_dtypes(include=['float64', 'int64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns
y = np.log1p(y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols)
    ])
model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', HuberRegressor(max_iter=2000, epsilon=1.35))
])
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_original_scale = np.expm1(y_pred)
y_test_original_scale = np.expm1(y_test)
print("R2 Score:", r2_score(y_test_original_scale, y_pred_original_scale))

R2 Score: 0.9715862711157408
